[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlospi2000/Notes_on_Computational_Physics/blob/main/10.Matrix_and_Fourier_Transform/A.Matrices_DFT-5.ipynb)

# Semana 10a — Matrices y transformada discreta de Fourier

**Física Computacional — 106018C** · Lenguaje: **Fortran y Python** · Capítulo 5: Matrices

Esta semana tiene dos notebooks:

| Notebook | Contenido |
|---|---|
| **10a** (este) | Una señal como suma de oscilaciones · operaciones con matrices · la DFT como producto matriz-vector · inversa · frecuencias, aliasing, leakage · costo · ejercicios de calentamiento 1 y 2 |
| **10b** | De la DFT a la serie de Fourier y a la transformada continua · fenómeno de Gibbs · ejercicio de calentamiento 3 · **Taller 10** |

Los tres **ejercicios de calentamiento**, en recuadros dorados, se trabajan en la clase del lunes. No se califican; preparan el Taller 10.

**Objetivos de este notebook**

1. Interpretar la transformada de Fourier como la descomposición de una señal en oscilaciones.
2. Realizar las operaciones básicas con matrices en NumPy y en Fortran: suma, producto, transpuesta y transpuesta conjugada.
3. Escribir la DFT como un producto matriz-vector y su inversa con la transpuesta conjugada.
4. Interpretar los coeficientes de la DFT: frecuencias, componente continua, frecuencia de Nyquist, simetría, aliasing y leakage.
5. Reconocer el costo $O(N^2)$ de la DFT matricial, que motiva la FFT de la Semana 11.

In [ ]:
# Preparación: bibliotecas de Python, colores del curso y herramientas de compilación
import time
import numpy as np
import matplotlib.pyplot as plt

AZUL, DORADO, ROJO, VERDE, GRIS, MORADO = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F", "#6A3D9A"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=4, suppress=True)

# F2PY compila con meson y ninja desde Python 3.12 (igual que en la Semana 9)
!pip -q install meson ninja
!gfortran --version | head -n 1
print("NumPy", np.__version__)

## 1. Una señal es una suma de oscilaciones

La figura siguiente muestra la idea central de la semana. En negro, una señal que se repite con período $T$ y que se parece a una onda cuadrada. Esa señal es la **suma** de las oscilaciones de colores, que se dibujan alejándose hacia el fondo: un seno que completa 1 ciclo por período, otro que completa 3, otro 5, y así hasta 11, cada uno con su propia amplitud.

La misma señal se puede describir de dos maneras:

- En el **dominio del tiempo**: el valor de la señal en cada instante $t$ (la curva negra).
- En el **dominio de la frecuencia**: la amplitud $b_n$ de cada oscilación que la compone (los puntos del plano de la derecha, uno por frecuencia $n/T$; las frecuencias pares tienen amplitud cero).

Las dos descripciones contienen la misma información. Pasar del tiempo a la frecuencia es el **análisis de Fourier**; volver de la frecuencia al tiempo, sumando las oscilaciones, es la **síntesis de Fourier**. La **transformada de Fourier** es la operación matemática que hace el análisis.

In [ ]:
# Figura: una señal en el dominio del tiempo y en el dominio de la frecuencia
# (dibujo en perspectiva; no es necesario estudiar este código)
from matplotlib.patches import Polygon

CIAN = "#1F8FB3"

# ---- proyección: punto (t, n, y) -> (X, Y) en el papel
UX, UY = 3.0, -0.95        # avance en el papel por cada segundo de t (hacia abajo a la derecha)
VX, VY = 0.42, 0.30        # avance por cada unidad de frecuencia n (hacia arriba a la derecha)
def P(t, n, y):
    t, n, y = np.broadcast_arrays(np.asarray(t, float), np.asarray(n, float), np.asarray(y, float))
    return t*UX + n*VX, t*UY + n*VY + y

T = 1.0                                      # período [s]
armonicos = [1, 3, 5, 7, 9, 11]
colores = ["#5B7FC7", VERDE, ROJO, DORADO, MORADO, CIAN]   # 1 ciclo en azul claro: la señal va en azul oscuro
b = {n: 4/(n*np.pi) for n in armonicos}      # coeficientes de la onda cuadrada
t = np.linspace(0, 2.25*T, 1500)
TF = 2.75                                    # posición del plano de la frecuencia (en t)

fig, ax = plt.subplots(figsize=(13, 7))
plt.rcParams.update({"mathtext.fontset": "cm"})
ax.set_aspect("equal"); ax.axis("off")

# ---- planos
def plano(esquinas, **kw):
    X, Y = zip(*[P(*e) for e in esquinas])
    ax.add_patch(Polygon(np.column_stack([X, Y]), closed=True, **kw))
plano([(-0.15, 0, -1.75), (2.5, 0, -1.75), (2.5, 0, 1.85), (-0.15, 0, 1.85)],
      facecolor="white", edgecolor=GRIS, lw=1, zorder=1)
plano([(TF, -0.8, -1.75), (TF, 13.2, -1.75), (TF, 13.2, 1.85), (TF, -0.8, 1.85)],
      facecolor="#F7F8FB", edgecolor=GRIS, lw=1, zorder=0)

# ---- oscilaciones: de la más lejana a la más cercana
for n, c in reversed(list(zip(armonicos, colores))):
    X0, Y0 = P([0, TF - 0.05], n, 0)                         # línea guía hacia el plano de la frecuencia
    ax.annotate("", xy=(X0[1], Y0[1]), xytext=(X0[0], Y0[0]),
                arrowprops=dict(arrowstyle="-|>", color="#C8C8C8", lw=1), zorder=2)
    X, Y = P(t, n, b[n]*np.sin(2*np.pi*n*t/T))
    ax.plot(X, Y, color=c, lw=1.8, alpha=0.9, zorder=3)

# ---- señal: suma de las oscilaciones, sobre el plano del tiempo
suma = sum(b[n]*np.sin(2*np.pi*n*t/T) for n in armonicos)
ax.plot(*P(t, 0, suma), color=AZUL, lw=3.2, zorder=6)

# ejes del plano del tiempo
X, Y = P([-0.1, 2.45], 0, 0)
ax.annotate("", xy=(X[1], Y[1]), xytext=(X[0], Y[0]), arrowprops=dict(arrowstyle="-|>", color="k", lw=1.6), zorder=5)
X, Y = P(0, 0, [-1.5, 1.6])
ax.annotate("", xy=(X[1], Y[1]), xytext=(X[0], Y[0]), arrowprops=dict(arrowstyle="-|>", color="k", lw=1.6), zorder=5)
ax.text(*P(0.07, 0, 1.6), r"$y$", fontsize=17, zorder=7)
ax.text(*P(2.42, 0, -0.5), r"$t$ [s]", fontsize=17, zorder=7)
for k, et in [(1, r"$T$"), (2, r"$2T$")]:
    ax.plot(*P([k*T]*2, 0, [-0.06, 0.06]), color="k", lw=1.5, zorder=7)
    ax.text(*P(k*T, 0, -0.62), et, fontsize=16, zorder=7, ha="center",
            bbox=dict(facecolor="white", edgecolor="none", pad=1))
xt, yt = P(1.0, 0, -1.5)
ax.text(xt, yt, "Dominio del tiempo", fontsize=18, rotation=np.degrees(np.arctan2(UY, UX)),
        ha="center", va="center", zorder=7)

# ---- plano de la frecuencia
X, Y = P(TF, [-0.5, 12.8], 0)
ax.annotate("", xy=(X[1], Y[1]), xytext=(X[0], Y[0]), arrowprops=dict(arrowstyle="-|>", color="k", lw=1.6), zorder=5)
X, Y = P(TF, 0, [-1.6, 1.6])
ax.annotate("", xy=(X[1], Y[1]), xytext=(X[0], Y[0]), arrowprops=dict(arrowstyle="-|>", color="k", lw=1.6), zorder=5)
ax.text(*P(TF, -0.4, 1.45), r"$b_n$", fontsize=17, ha="right")
ax.text(*P(TF, 12.9, 0.1), r"$f_n$ [Hz]", fontsize=17)
nn = np.linspace(0.8, 12, 200)
ax.plot(*P(TF, nn, 4/(nn*np.pi)), "--", color="#9AA3C7", lw=1.2, zorder=4)   # envolvente 4/(pi n)
for n in range(1, 13):
    ax.plot(*P(TF, [n]*2, [-0.07, 0.07]), color="k", lw=1.3, zorder=5)
    if n % 2 == 0:
        ax.plot(*P(TF, n, 0), "o", color=AZUL, ms=6, zorder=6)                  # b_n = 0 para n par
for n, c in zip(armonicos, colores):
    ax.plot(*P(TF, [n]*2, [0, b[n]]), ":", color=c, lw=1.5, zorder=5)
    ax.plot(*P(TF, n, b[n]), "o", color=c, ms=9, zorder=6)
    ax.text(*P(TF, n - 0.15, -0.55), rf"$\frac{{{n}}}{{T}}$", fontsize=17, ha="center")
xt, yt = P(TF, 7.5, -1.2)
ax.text(xt, yt, "Dominio de la frecuencia", fontsize=19, rotation=np.degrees(np.arctan2(VY, VX)),
        ha="center", va="center")

ax.autoscale_view()
plt.show()

### 1.1 ¿Cómo se mide cuánto hay de cada oscilación?

Si la señal está muestreada en $N$ puntos, $x_0, \dots, x_{N-1}$, y se quiere saber cuánto contiene de una oscilación de $j$ ciclos, $c_k = \cos(2\pi jk/N)$, se multiplican las dos muestra por muestra y se suma:

$$\sum_{k=0}^{N-1} x_k\, c_k .$$

Es el **producto escalar** de dos vectores. Si la señal contiene esa oscilación, los productos son casi siempre positivos y la suma es grande. Si no la contiene, los productos positivos y negativos se cancelan y la suma es casi cero. La figura lo muestra con una señal que contiene 6 ciclos en sus 64 muestras: el producto escalar con la oscilación de 6 ciclos da 32, y con la de 5 ciclos da prácticamente 0.

Hacer este producto escalar para **todas** las oscilaciones $j = 0, 1, \dots, N-1$ es multiplicar una **matriz**, cuyas filas son las oscilaciones, por el **vector** de muestras. Por eso la transformada de Fourier aparece en el capítulo de matrices.

In [ ]:
# Figura: el producto escalar con cada oscilación mide cuánto de ella contiene la señal
N, dt = 64, 1/32                               # 64 muestras en T = 2 s
k = np.arange(N)
tk = k*dt
x = np.cos(2*np.pi*3*tk) + 0.4*np.cos(2*np.pi*7*tk)   # 3 Hz = 6 ciclos en 2 s; 7 Hz = 14 ciclos

fig, ax = plt.subplots(2, 2, figsize=(10, 5), sharex=True, sharey="row")
for col, j in enumerate([6, 5]):
    c = np.cos(2*np.pi*j*k/N)                  # oscilación de j ciclos
    producto = x*c
    ax[0, col].plot(tk, x, "o-", color=AZUL, ms=2.5, lw=1, label="señal x_k")
    ax[0, col].plot(tk, c, "--", color=DORADO, lw=1.5, label=f"oscilación de {j} ciclos")
    ax[0, col].legend(loc="upper right", fontsize=8)
    ax[1, col].bar(tk, producto, width=0.8*dt, color=np.where(producto > 0, VERDE, ROJO))
    ax[1, col].set_title(f"producto x_k c_k:  suma = {abs(round(producto.sum(), 2)):.2f}", fontsize=10)
    ax[1, col].set_xlabel("t [s]")
ax[0, 0].set_title("La señal contiene 6 ciclos", fontsize=10)
ax[0, 1].set_title("La señal no contiene 5 ciclos", fontsize=10)
plt.tight_layout(); plt.show()

## 2. Matrices en NumPy y en Fortran

Una matriz $A$ de $n$ filas y $m$ columnas tiene elementos $a_{ij}$, donde $i$ es el índice de fila y $j$ el de columna. En los dos lenguajes se guarda como un arreglo de dos dimensiones. Las operaciones que se usan esta semana son:

| Operación | Definición | NumPy | Fortran |
|---|---|---|---|
| Suma | $c_{ij} = a_{ij} + b_{ij}$ (misma forma) | `A + B` | `A + B` |
| Producto por escalar | $c_{ij} = \alpha\, a_{ij}$ | `alpha*A` | `alpha*A` |
| Producto matricial | $c_{ij} = \sum_{k} a_{ik}\, b_{kj}$ | `A @ B` | `matmul(A, B)` |
| Producto matriz-vector | $y_i = \sum_{k} a_{ik}\, x_k$ | `A @ x` | `matmul(A, x)` |
| Transpuesta | $(A^T)_{ij} = a_{ji}$ | `A.T` | `transpose(A)` |
| Transpuesta conjugada | $(A^H)_{ij} = \overline{a_{ji}}$ | `A.conj().T` | `conjg(transpose(A))` |
| Matriz identidad | $\delta_{ij}$ | `np.eye(n)` | se construye con un ciclo |

En el producto matricial, el elemento $c_{ij}$ es el producto escalar de la **fila** $i$ de $A$ con la **columna** $j$ de $B$; por eso el número de columnas de $A$ debe ser igual al número de filas de $B$. Un producto de matrices de $n\times n$ requiere $n^3$ multiplicaciones; un producto matriz-vector, $n^2$.

En la transpuesta conjugada, $\overline{a}$ es el complejo conjugado de $a$. Para una matriz real, $A^H = A^T$.

In [ ]:
# Figura: el elemento c_ij del producto es la fila i de A por la columna j de B
A = np.arange(1, 7).reshape(2, 3).astype(float)
B = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, -1.0]])
Cm = A @ B
i, j = 1, 0                                    # elemento que se resalta

fig, ax = plt.subplots(1, 3, figsize=(9, 2.6), gridspec_kw={"width_ratios": [3, 2, 2]})
for a_, M, nombre in zip(ax, [A, B, Cm], ["A (2×3)", "B (3×2)", "C = A B (2×2)"]):
    resalte = np.zeros(M.shape)
    if nombre.startswith("A"): resalte[i, :] = 1
    if nombre.startswith("B"): resalte[:, j] = 1
    if nombre.startswith("C"): resalte[i, j] = 1
    a_.imshow(resalte, cmap="YlOrBr", vmin=0, vmax=2)
    for (r, c), v in np.ndenumerate(M):
        a_.text(c, r, f"{v:g}", ha="center", va="center", fontsize=12)
    a_.set_title(nombre); a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)
fig.suptitle(f"c[{i},{j}] = fila {i} de A · columna {j} de B = 4·1 + 5·0 + 6·2 = {Cm[i, j]:g}", y=1.02)
plt.tight_layout(); plt.show()

**Diferencias que importan**

- NumPy numera desde 0 y Fortran desde 1: el elemento $a_{11}$ es `A[0, 0]` en NumPy y `A(1,1)` en Fortran.
- `A * B` es el producto **elemento a elemento** en los dos lenguajes, no el producto matricial.
- Fortran guarda las matrices por columnas y NumPy por filas (Semana 9). En Fortran, el ciclo más interno debe recorrer el **primer** índice.

### 2.1 Producto matricial con ciclos en Fortran

El módulo siguiente programa el producto matricial con tres ciclos, para ver la definición, y la transpuesta conjugada con las funciones intrínsecas. Se compila como librería con F2PY y se compara con NumPy.

In [ ]:
%%writefile modulo_matrices.f90
! ============================================================
! Módulo:    matrices
! Propósito: producto matricial con ciclos y transpuesta
!            conjugada, para usar desde Python con F2PY.
! ============================================================
module matrices
  implicit none
contains

  ! producto: c = a b, con la definición c(i,j) = sum_k a(i,k) b(k,j)
  ! Entradas: a(n,m), b(m,p)
  ! Salida:   c(n,p)
  subroutine producto(n, m, p, a, b, c)
    integer, intent(in)  :: n, m, p
    real(8), intent(in)  :: a(n,m), b(m,p)
    real(8), intent(out) :: c(n,p)
    integer :: i, j, k
    c = 0.0d0
    do j = 1, p                       ! columna de c
       do k = 1, m                    ! índice de la suma
          do i = 1, n                 ! fila de c: ciclo interno sobre el primer índice
             c(i,j) = c(i,j) + a(i,k)*b(k,j)
          end do
       end do
    end do
  end subroutine producto

  ! adjunta: transpuesta conjugada de una matriz compleja
  ! Entrada: a(n,m)
  ! Salida:  ah(m,n)
  subroutine adjunta(n, m, a, ah)
    integer,    intent(in)  :: n, m
    complex(8), intent(in)  :: a(n,m)
    complex(8), intent(out) :: ah(m,n)
    ah = conjg(transpose(a))
  end subroutine adjunta

end module matrices

In [ ]:
# Compila la librería "mat" y muestra la firma del producto
!python -m numpy.f2py -c modulo_matrices.f90 -m mat 2>&1 | tail -n 1
import mat
print(mat.matrices.producto.__doc__)

In [ ]:
# Comparación con NumPy: las matrices A y B de la figura anterior
print("Fortran:\n", mat.matrices.producto(A, B))
print("NumPy:\n", A @ B)

# Transpuesta conjugada de una matriz compleja 2x2
Z = np.array([[1 + 1j, 2],
              [3j,     4 - 2j]])
print("Fortran:\n", mat.matrices.adjunta(Z))
print("NumPy:\n", Z.conj().T)

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

**Lectura de la firma.** F2PY deduce `n`, `m` y `p` de las formas de `a` y `b`, de modo que se llama `producto(A, B)`. La matriz de salida `c` regresa como arreglo de NumPy. Si las dimensiones no son compatibles, F2PY lo detecta antes de entrar a Fortran.

</div>

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 1 (no calificado): matriz de rotación en Fortran**

La matriz que rota un vector del plano un ángulo $\theta$ en sentido antihorario es

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}.$$

Es una matriz **ortogonal**: su inversa es su transpuesta, $R\,R^T = I$. Es la versión real de lo que se verá para la matriz de Fourier, cuya inversa es su transpuesta conjugada (sección 4).

En `ejercicio_rotacion.f90`, con $\theta = 30^\circ$ y $\mathbf{v} = (1, 0)$:

1. Construya $R(\theta)$.
2. Obtenga $R\,\mathbf{v}$.
3. Aplique $R$ doce veces seguidas a $\mathbf{v}$: doce rotaciones de $30^\circ$ son una vuelta completa.
4. Compruebe la ortogonalidad con el máximo de $|R\,R^T - I|$.

**Debe obtener:** $R\,\mathbf{v} = (0.866025,\ 0.500000)$; después de doce rotaciones, una diferencia máxima con $\mathbf{v}$ del orden de $10^{-15}$; y $\max|R\,R^T - I|$ del orden de $10^{-16}$ o menor.

</div>

In [ ]:
%%writefile ejercicio_rotacion.f90
! ============================================================
! Programa:  ejercicio_rotacion   (Ejercicio de calentamiento 1)
! Propósito: matriz de rotación en el plano: producto
!            matriz-vector, potencias y ortogonalidad.
! Salida:    en pantalla
! ============================================================
program ejercicio_rotacion
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  real(dp), parameter :: pi = 4*atan(1.0_dp)
  real(dp) :: R(2,2), v(2), w(2), identidad(2,2), theta
  integer :: i

  theta = 30.0_dp*pi/180.0_dp               ! ángulo de rotación [rad]
  v = [1.0_dp, 0.0_dp]                      ! vector inicial
  identidad = 0.0_dp                        ! matriz identidad de 2 x 2
  identidad(1,1) = 1.0_dp
  identidad(2,2) = 1.0_dp

  ! TODO 1: construya la matriz de rotación R(theta)
  ! TODO 2: imprima R v
  ! TODO 3: aplique R doce veces a v e imprima la diferencia máxima con v
  ! TODO 4: imprima el máximo de |R R^T - I|
end program ejercicio_rotacion

In [ ]:
!gfortran -fcheck=bounds -o ejercicio_rotacion ejercicio_rotacion.f90 && ./ejercicio_rotacion

## 3. La transformada discreta de Fourier como producto matriz-vector

Se tiene una señal muestreada en $N$ instantes equiespaciados, $x_k = x(t_0 + k\,\Delta t)$, con $k = 0, 1, \dots, N-1$. La **transformada discreta de Fourier** de la señal es el conjunto de $N$ números complejos

$$X_j = \sum_{k=0}^{N-1} x_k\, e^{-2\pi i\, jk/N}, \qquad j = 0, 1, \dots, N-1,$$

donde

- $x_k$ es la muestra $k$ de la señal (real o compleja);
- $X_j$ es el **coeficiente de Fourier** $j$: su módulo $|X_j|$ mide cuánto contribuye a la señal una oscilación que completa $j$ ciclos en las $N$ muestras, y su argumento, la fase de esa oscilación;
- $e^{-2\pi i\, jk/N} = \cos(2\pi jk/N) - i\,\sin(2\pi jk/N)$ es la oscilación discreta de $j$ ciclos evaluada en la muestra $k$; incluye un coseno y un seno para detectar la oscilación con cualquier fase;
- $i$ es la unidad imaginaria.

Es el producto escalar de la sección 1.1, hecho para cada $j$. Con $\omega = e^{-2\pi i/N}$, cada término es $\omega^{jk}\,x_k$, y las $N$ ecuaciones juntas forman un **producto matriz-vector**:

$$
\begin{pmatrix} X_0 \\ X_1 \\ X_2 \\ \vdots \\ X_{N-1} \end{pmatrix}
=
\begin{pmatrix}
1 & 1 & 1 & \cdots & 1 \\
1 & \omega & \omega^2 & \cdots & \omega^{N-1} \\
1 & \omega^2 & \omega^4 & \cdots & \omega^{2(N-1)} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & \omega^{N-1} & \omega^{2(N-1)} & \cdots & \omega^{(N-1)^2}
\end{pmatrix}
\begin{pmatrix} x_0 \\ x_1 \\ x_2 \\ \vdots \\ x_{N-1} \end{pmatrix},
\qquad \mathbf{X} = W\,\mathbf{x},
$$

donde $W$ es la **matriz de Fourier**, de elementos $W_{jk} = \omega^{jk}$. $W$ es simétrica ($W^T = W$) y sus elementos tienen módulo 1.

In [ ]:
def matriz_dft(N):
    """Matriz de Fourier W de N x N, con W[j, k] = exp(-2 pi i j k / N)."""
    k = np.arange(N)
    return np.exp(-2j*np.pi*np.outer(k, k)/N)     # np.outer(k, k)[j, k] = j*k

In [ ]:
# Figura: la matriz de Fourier y algunas de sus filas
N = 16
W = matriz_dft(N)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [1, 1.5]})
im = ax[0].imshow(W.real, cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set(title="Re W  (N = 16)", xlabel="columna k (muestra)", ylabel="fila j (oscilación)")
ax[0].grid(False); fig.colorbar(im, ax=ax[0], shrink=0.8)
for desplazamiento, (j, c) in enumerate(zip([0, 1, 2, 8], [GRIS, AZUL, VERDE, ROJO])):
    ax[1].plot(np.arange(N), W[j].real - 2.6*desplazamiento, "o-", color=c, ms=4, label=f"fila j = {j}")
ax[1].set(title="Parte real de cuatro filas", xlabel="k", yticks=[])
ax[1].legend(loc="upper right", fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

Cada **fila** de $W$ es una oscilación muestreada: la fila 0 es constante, la fila 1 completa un ciclo en las 16 muestras, la fila 2 completa dos, y la fila $N/2 = 8$ cambia de signo en cada muestra. Multiplicar $W$ por el vector de la señal hace, de una sola vez, el producto escalar de la señal con cada una de esas oscilaciones.

### 3.1 Ejemplo con $N = 4$

In [ ]:
# Ejemplo con N = 4: omega = exp(-2 pi i/4) = -i
W4 = matriz_dft(4)
print("W (N = 4):\n", W4)

x = np.array([1.0, 2.0, 3.0, 4.0])
X = W4 @ x
print("\nx =", x)
print("X =", X)

Para $N = 4$, $\omega = e^{-i\pi/2} = -i$, y los elementos de $W$ son potencias de $-i$: $1, -i, -1, i$. Con $x = (1, 2, 3, 4)$:

- $X_0 = 1 + 2 + 3 + 4 = 10$: el coeficiente $j = 0$ es la **suma** de las muestras; $X_0/N = 2.5$ es el promedio o **componente continua**.
- $X_1 = 1 - 2i - 3 + 4i = -2 + 2i$.
- $X_2 = 1 - 2 + 3 - 4 = -2$: la oscilación más rápida posible alterna el signo en cada muestra.
- $X_3 = -2 - 2i$, el complejo conjugado de $X_1$.

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Para verificar resultados.** NumPy calcula la misma transformada con `np.fft.fft(x)`, que usa el algoritmo rápido de la Semana 11. En estos notebooks se usa solo como **referencia** para comprobar la construcción matricial.

</div>

In [ ]:
# Comprobación con la función de NumPy, solo como referencia
x = np.random.default_rng(2026).normal(size=64)
print("máx |W x - fft(x)| =", np.max(np.abs(matriz_dft(64) @ x - np.fft.fft(x))))

## 4. La transformada inversa y la transpuesta conjugada

La señal se recupera a partir de sus coeficientes con la **transformada inversa**, que es la síntesis de Fourier: sumar las oscilaciones, cada una con su coeficiente.

$$x_k = \frac{1}{N}\sum_{j=0}^{N-1} X_j\, e^{+2\pi i\, jk/N}, \qquad k = 0, 1, \dots, N-1,$$

Difiere de la directa en el signo del exponente y en el factor $1/N$. En forma matricial, el signo opuesto equivale a conjugar los elementos de $W$, y como $W$ es simétrica, $\overline{W} = W^H$:

$$\mathbf{x} = \frac{1}{N}\, W^H\, \mathbf{X}, \qquad \text{es decir,} \qquad W^{-1} = \frac{1}{N}\, W^H .$$

La inversa de la matriz de Fourier **no** se calcula resolviendo un sistema de ecuaciones: es su transpuesta conjugada dividida por $N$. Eso se debe a que las filas de $W$ son **ortogonales**: el producto escalar de dos oscilaciones distintas es cero (sección 1.1), y el de una oscilación consigo misma es $N$,

$$\sum_{k=0}^{N-1} \omega^{jk}\,\overline{\omega^{lk}} = N\,\delta_{jl} \qquad\Longleftrightarrow\qquad W\, W^H = N\, I ,$$

donde $I$ es la matriz identidad.

In [ ]:
N = 16
W = matriz_dft(N)
P = W @ W.conj().T                        # productos escalares entre filas

# Figura: W W^H es N veces la identidad
fig, ax = plt.subplots(figsize=(4, 3.4))
im = ax.imshow(np.abs(P), cmap="Blues")
ax.set(title="|W W^H|  (N = 16)", xlabel="fila l", ylabel="fila j"); ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.85); plt.tight_layout(); plt.show()

print("máx |W W^H - N I| =", np.max(np.abs(P - N*np.eye(N))))

# Ida y vuelta: x -> X -> x
x = np.random.default_rng(1).normal(size=N)
x_rec = (W.conj().T @ (W @ x)) / N
print("máx |x_rec - x|   =", np.max(np.abs(x_rec - x)))

<div style="border-left: 5px solid #162A54; background-color: #EEF1F7; padding: 10px 14px; margin: 8px 0;">

**Normalización unitaria.** Algunos textos definen la matriz $F = W/\sqrt{N}$. Con esa elección, $F F^H = I$: $F$ es una **matriz unitaria**, y la inversa es simplemente $F^H$, sin factor $1/N$. Las dos convenciones dan la misma información; en este curso se usa la de NumPy, con $W$ en la directa y $1/N$ en la inversa.

</div>

## 5. Frecuencias, simetría, aliasing y leakage

### 5.1 La frecuencia de cada coeficiente

Las $N$ muestras cubren una duración $T = N\,\Delta t$. El coeficiente $X_j$ corresponde a una oscilación de $j$ ciclos en ese tiempo, es decir, a la frecuencia

$$f_j = \frac{j}{N\,\Delta t} = \frac{j}{T}, \qquad \Delta f = \frac{1}{T},$$

donde $\Delta f$ es la **resolución en frecuencia**: dos frecuencias más cercanas que $\Delta f$ no se distinguen. La frecuencia más alta que se puede representar es la **frecuencia de Nyquist**,

$$f_N = \frac{1}{2\,\Delta t},$$

que corresponde a $j = N/2$: una oscilación que cambia de signo en cada muestra (la fila 8 de la figura de la sección 3).

### 5.2 Simetría para señales reales

Si las muestras $x_k$ son reales, los coeficientes cumplen

$$X_{N-j} = \overline{X_j}.$$

Los coeficientes con $j > N/2$ no aportan información nueva: representan las **frecuencias negativas** $f = -(N-j)/T$. La figura siguiente muestra a qué frecuencia corresponde cada posición del arreglo $X$.

In [ ]:
# Figura: frecuencia asociada a cada índice j del arreglo X (N = 16)
N = 16
j = np.arange(N)
f_j = np.where(j <= N//2, j, j - N)            # frecuencia en unidades de Δf = 1/T
colores = [GRIS if jj == 0 else ROJO if jj == N//2 else DORADO if jj < N//2 else AZUL for jj in j]

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(j, f_j, color=colores)
for jj, ff in zip(j, f_j):
    ax.text(jj, ff + (0.4 if ff >= 0 else -0.9), f"{ff:+d}" if ff else "0", ha="center", fontsize=8)
ax.set(xlabel="índice j en el arreglo X", ylabel="frecuencia  f_j / Δf", xticks=j, ylim=(-9.5, 11.5))
ax.text(3.5, 8.6, "frecuencias positivas", color=DORADO, ha="center")
ax.text(12, -8.8, "frecuencias negativas", color=AZUL, ha="center")
ax.text(8, 10.2, "Nyquist", color=ROJO, ha="center")
plt.tight_layout(); plt.show()

Las frecuencias bajas, positivas y negativas, están en los **dos extremos** del arreglo; las altas, en el **centro**, alrededor de $j = N/2$. Por eso el espectro de una señal real se grafica solo para $j = 0, \dots, N/2$. Una componente $A\cos(2\pi f_j t + \varphi)$ de la señal aparece repartida entre $X_j$ y $X_{N-j}$, cada uno con módulo $AN/2$; la amplitud se recupera como

$$A_j = \frac{2\,|X_j|}{N}\quad (0 < j < N/2), \qquad A_0 = \frac{|X_0|}{N}.$$

In [ ]:
# Señal de prueba: componente continua + dos cosenos
#   x(t) = 0.5 + 1.0 cos(2 pi 3 t) + 0.4 cos(2 pi 7 t)
dt = 1/32                    # paso de muestreo [s]: 32 muestras por segundo
N  = 64                      # número de muestras: T = N dt = 2 s
t  = np.arange(N)*dt
x  = 0.5 + 1.0*np.cos(2*np.pi*3*t) + 0.4*np.cos(2*np.pi*7*t)

X = matriz_dft(N) @ x
f = np.arange(N)/(N*dt)      # frecuencia de cada coeficiente [Hz]
A = 2*np.abs(X)/N            # amplitud de cada componente
A[0] = np.abs(X[0])/N        # la componente continua no se reparte

print(f"Resolución Δf = {1/(N*dt)} Hz,  Nyquist f_N = {1/(2*dt)} Hz")
for j in np.where(A > 1e-10)[0]:
    if j <= N//2:
        print(f"  j = {j:2d}   f = {f[j]:5.2f} Hz   amplitud = {A[j]:.4f}")
print("simetría: máx |X[N-j] - conj(X[j])| =", np.max(np.abs(X[1:][::-1] - X[1:].conj())))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(t, x, "o-", color=AZUL, ms=3)
ax[0].set(xlabel="t [s]", ylabel="x(t)", title="Dominio del tiempo")
ax[1].stem(f[:N//2 + 1], A[:N//2 + 1], linefmt=DORADO, markerfmt="o", basefmt=GRIS)
ax[1].set(xlabel="f [Hz]", ylabel="amplitud", title="Dominio de la frecuencia")
plt.tight_layout(); plt.show()

### 5.3 Aliasing

La DFT es exacta como transformación entre $N$ números y otros $N$ números. Los problemas aparecen cuando se interpreta como el espectro de la señal continua de la que provienen las muestras.

Una componente de frecuencia mayor que $f_N$ produce exactamente las mismas muestras que una de frecuencia más baja, y la DFT la asigna a esa frecuencia baja. Con 32 muestras por segundo, un coseno de 25 Hz da las mismas muestras que uno de $32 - 25 = 7$ Hz. La única solución es muestrear más rápido o filtrar la señal antes de muestrearla.

In [ ]:
# Figura: aliasing. 25 Hz y 7 Hz coinciden en las muestras tomadas a 32 Hz
tc = np.linspace(0, 0.5, 2000)               # tiempo "continuo" [s]
tm = np.arange(0, 0.5, 1/32)                 # instantes de muestreo
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(tc, np.cos(2*np.pi*25*tc), color=ROJO, lw=0.8, label="25 Hz")
ax.plot(tc, np.cos(2*np.pi*7*tc), color=AZUL, lw=1.5, label="7 Hz")
ax.plot(tm, np.cos(2*np.pi*7*tm), "o", color="k", ms=5, label="muestras (32 por segundo)")
ax.set(xlabel="t [s]", ylabel="x(t)", title="Dos frecuencias, las mismas muestras")
ax.legend(loc="lower right", fontsize=8, ncol=3); plt.tight_layout(); plt.show()
print("máx |x25 - x7| en las muestras =", np.max(np.abs(np.cos(2*np.pi*25*tm) - np.cos(2*np.pi*7*tm))))

### 5.4 Leakage (fuga espectral)

La DFT trata las $N$ muestras como **un período** de una señal que se repite. Si la duración $T$ contiene un número entero de ciclos de una componente, la repetición empalma sin saltos. Si no, aparece un salto en cada empalme, y ese salto, que no está en la señal original, necesita muchas frecuencias para representarse: la componente se reparte entre muchos coeficientes.

In [ ]:
# Figura: cómo "ve" la DFT las muestras, repetidas una y otra vez
fig, ax = plt.subplots(2, 1, figsize=(9, 4.2), sharex=True)
for a_, f0, color in zip(ax, [3.0, 3.25], [VERDE, ROJO]):
    x1 = np.cos(2*np.pi*f0*t)                    # las N = 64 muestras de T = 2 s
    rep = np.tile(x1, 3)                          # tres repeticiones
    a_.plot(np.arange(3*N)*dt, rep, ".-", color=color, ms=3, lw=0.8)
    for m in [1, 2]:
        a_.axvline(m*N*dt, color=GRIS, ls="--", lw=1)
    a_.set(ylabel="x", title=f"{f0} Hz: {f0*N*dt:g} ciclos en T = 2 s")
ax[1].set_xlabel("t [s]")
plt.tight_layout(); plt.show()

Con 3 Hz caben 6 ciclos completos y la repetición es una señal continua. Con 3.25 Hz caben 6.5 ciclos y en cada empalme hay un salto. El leakage se reduce multiplicando las muestras por una **ventana** que las lleva suavemente a cero en los extremos, como la de Hann, $w_k = \tfrac{1}{2}\left[1 - \cos(2\pi k/N)\right]$, que elimina el salto.

In [ ]:
# Espectro del coseno de 3.25 Hz, sin ventana y con ventana de Hann
x_fuga = np.cos(2*np.pi*3.25*t)
ventana = 0.5*(1 - np.cos(2*np.pi*np.arange(N)/N))      # ventana de Hann
W64 = matriz_dft(N)
A_fuga = 2*np.abs(W64 @ x_fuga)/N
A_hann = 2*np.abs(W64 @ (x_fuga*ventana))/N / 0.5        # 0.5: promedio de la ventana

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.semilogy(f[:N//2 + 1], A_fuga[:N//2 + 1], "o-", color=ROJO, ms=3, label="sin ventana")
ax.semilogy(f[:N//2 + 1], A_hann[:N//2 + 1], "s-", color=VERDE, ms=3, label="ventana de Hann")
ax.set(xlabel="f [Hz]", ylabel="amplitud", title="Leakage de un coseno de 3.25 Hz", ylim=(1e-5, 2))
ax.legend(); plt.tight_layout(); plt.show()

<div style="border: 2px solid #C9972C; background-color: #FBF6EA; padding: 10px 14px; margin: 8px 0;">

**Ejercicio de calentamiento 2 (no calificado): la DFT con dos ciclos**

La celda siguiente tiene $N = 8$ muestras de la señal

$$x_k = 1 + 2\cos\left(\frac{2\pi k}{8}\right) + \sin\left(\frac{2\pi\, 3k}{8}\right), \qquad k = 0, \dots, 7.$$

1. Complete la función `dft_ciclos`, que obtiene cada $X_j$ con la suma de la sección 3, usando un ciclo sobre $j$ y otro sobre $k$, sin la matriz $W$.
2. Compare su resultado con `matriz_dft(8) @ x`.
3. Obtenga la amplitud de cada componente con las fórmulas de la sección 5.2.
4. Explique por qué $X_1$ es real y $X_3$ es imaginario puro.

**Debe obtener:** $X = (8,\ 8,\ 0,\ -4i,\ 0,\ 4i,\ 0,\ 8)$, con un error frente a la matriz menor que $10^{-12}$, y amplitudes $A_0 = 1$, $A_1 = 2$ y $A_3 = 1$, que son los coeficientes de la señal.

</div>

In [ ]:
# Ejercicio de calentamiento 2
N = 8
k = np.arange(N)
x = 1 + 2*np.cos(2*np.pi*k/N) + np.sin(2*np.pi*3*k/N)

def dft_ciclos(x):
    '''DFT por la definición, con un ciclo sobre j y otro sobre k.'''
    N = len(x)
    X = np.zeros(N, dtype=complex)
    # TODO: obtenga cada X[j] con la suma de la sección 3
    return X

X = dft_ciclos(x)
# TODO: imprima X, el error máximo frente a matriz_dft(N) @ x y la amplitud
#       de cada componente para j = 0, ..., N/2 (sección 5.2)

## 6. Costo de la DFT matricial

El producto $W\mathbf{x}$ requiere $N^2$ multiplicaciones complejas, y la matriz ocupa $N^2$ números complejos de 16 bytes. Si $N$ se duplica, el tiempo y la memoria se multiplican por 4. La celda siguiente mide el tiempo de construir $W$ y multiplicarla por un vector.

In [ ]:
def mejor_tiempo(funcion, repeticiones=3):
    """Menor tiempo de ejecución [s] entre varias repeticiones (Semana 9)."""
    tiempos = []
    for _ in range(repeticiones):
        t0 = time.perf_counter()
        funcion()
        tiempos.append(time.perf_counter() - t0)
    return min(tiempos)

tamanos = [256, 512, 1024, 2048, 4096]
tiempos = []
print("    N    memoria de W     tiempo")
for N in tamanos:
    x = np.random.default_rng(N).normal(size=N)
    tiempos.append(mejor_tiempo(lambda: matriz_dft(N) @ x))
    print(f"{N:5d}   {16*N**2/2**20:8.1f} MiB   {tiempos[-1]*1e3:8.2f} ms")

# Figura: tiempo frente a N en escala logarítmica, con la referencia N^2
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.loglog(tamanos, np.array(tiempos)*1e3, "o-", color=AZUL, label="DFT matricial")
ax.loglog(tamanos, tiempos[-1]*1e3*(np.array(tamanos)/tamanos[-1])**2, "--", color=GRIS, label="∝ N²")
ax.set(xlabel="N", ylabel="tiempo [ms]", title="Costo O(N²)"); ax.legend()
ax.set_xticks(tamanos, [str(n) for n in tamanos]); ax.minorticks_off()
plt.tight_layout(); plt.show()

Los puntos siguen la recta $\propto N^2$. Una señal de audio de un segundo tiene 44 100 muestras: su matriz de Fourier ocuparía unos 29 GiB. La **transformada rápida de Fourier** (FFT) de la Semana 11 obtiene los mismos $N$ coeficientes en $O(N\log_2 N)$ operaciones y sin construir la matriz, aprovechando que $W$ solo tiene $N$ valores distintos, que se repiten (se ve en la figura de la sección 3).

**Continúa en el notebook 10b**: de la DFT a la serie de Fourier y a la transformada continua, el fenómeno de Gibbs y el Taller 10.